# ***Tarea 6. TRS y CDS***
### Luis Eduardo Jiménez del Muro - 30/04/2025
---

# **Total Return Swaps**

**Escribe una función en python llamada TRS**, la cual calcule el P&L (Profit and Loss) de un total return swap.
La función recibirá los siguientes inputs:

+ `n_bonds` : np.array que recibe la cantidad de bonos que se tienen de cada bono de la cartera sobre la cual esta el TRS.
+ `coupons` : np.array que recibe la tasa cupón de los bonos de la cartera sobre la cual esta el TRS.
+ `initial_prices` : np.array con los precios iniciales de los bonos
+ `final_prices` : np.array con los precios finales de los bonos
+ `fixed_rate` : Valor float con la tasa fija que paga el Protection Seller del Swap
+ `t` : Número de períodos al año en los que se intercambian los flujos del subyacente, si es anual es uno, si es semestral es dos.
+ `pay_total_return` : Booleano que indica si se esta calculando el P&L siendo Protection Buyer o Protection Seller.
Con las fórmulas dadas en clase y los inputs anteriores, realiza la función para que funcione sobre cualquier input de manera correcta.
A continuación puedes observar una idea de la estructura de la función, puedes utilizar los ejercicios y resultados vistos en la clase de
TRS para comprobar tu función.

In [1]:
import numpy as np

In [2]:
def TRS(n_bonds, coupons, initial_prices, final_prices, fixed_rate, t, pay_total_return):
    initial_pos = np.sum(n_bonds * initial_prices)
    final_pos = np.sum(n_bonds * final_prices)
    cupones = np.sum(n_bonds*coupons*100/t)
    total_return = final_pos - initial_pos + cupones
    tasa = fixed_rate * initial_pos / t
    return tasa - total_return if pay_total_return else total_return - tasa

In [3]:
n_bonds = np.array([35_000, 45_000, 50_000])
coupons = np.array([0.06, 0.07, 0.08])
initial_prices = np.array([98.75, 101.5, 97.6])
final_prices = np.array([96.4, 97.3, 92.8])
fixed_rate = 0.04
t = 2
pay_total_return = True

TRS(n_bonds, coupons, initial_prices, final_prices, fixed_rate, t, pay_total_return)

306825.0

# **Credit Default Swaps**

El spread de un CDS, es la tasa justa que se debería de pactar en un credit default swap en un momento dado en el tiempo, en función de la probabilidad de default y la tasa de recuperación, y es calculado de la siguiente manera:

$$
S_0 = \frac{\text{Protection Leg}}{\text{RPV01}}
$$

donde el Protection Leg es calculado de la siguiente manera:

$$
\text{Protection Leg} = \int_{0}^{T} Z(\tau)(1 - R)dPD(\tau)\
$$

y el RPV01 es calculado:

$$
\text{RPV01} = \sum_{j=1}^{N} Z(t_j)\Delta(t_{j-1}, t_j, B)Q(t_j)
$$

Con esta información, realiza una investigación exhaustiva acerca de los siguientes puntos:

### *¿Qué es el RPV01? ¿Cómo se interpreta? ¿Qué representan los elementos utilizados para calcularlos?*

El **Risky PV01**, o tambien llamado **RPV01** es el valor presente esperado de un basis point que sea pagado en la premium leg hasta el vencimiento del CDS o en caso de un evento crediticio. Es llamado *risky* debido a que se trata del valor esperado de flujos que no se sabe si sucederán. (pag: 3, 4)

La interpretación que tiene el RPV01 es la sensibilidad que tiene el premium leg ante el cambio de un basis point en el spread. Por lo que podríamos decir que a medida que crece el RPV01, el valor presente de las primas será mayor, lo que incrementaría el valor de la posición en el CDS. Los elementos que son utilizados para calcularlo representan lo siguiente:

$$
\text{RPV01} = \sum_{j=1}^{N} Z(t_j)\Delta(t_{j-1}, t_j, B)Q(t_j)
$$

Donde:

+ $Z(t_j)$: Es el factor de descuento de tasa Libor utilizada para calcular el valor presente de los flujos.

+ $\Delta(t_{j-1}, t_j, B)$: Es el número de días (representado en fracción de año) entre las fechas de pago de las primas.

+ $Q(t_j)$: Probabilidad de supervivencia de la entidad. Esta probabilidad incluye el riesgo de que la entidad sufra un evento crediticio deste el tiempo de valuacion hasta el tiempo de pago de la prima.

(pag. 7)

### *¿Qué es el Protection Leg? ¿Como se interpreta? ¿Que representan los elementos utilizados para calcularlos?*

El protection leg (1-RR) es el pago que se realiza en un CDS de parte del protection seller al proyection buyer en caso de que ocurra un evento crediticio en la entidad en cuestión. Este concepto se puede interpretar como el *seguro* que recibe el comprador en caso de que ocurra el evento crediticio a cambio de pagar su parte (primas o premium leg). (pag. 8, 9)

Además esto tambien representa el riesgo de crédito que tiene la entidad, por ejemplo: si el protection leg es de 0.9, quiere decir que la entidad solo pagaría el 10%, lo que indica, que podría traducirse a una mayor probabilidad de default, y por ende, un menor recovery rate. Por otro lado, es calculada como el valor esperado de la pérdida en caso del evento crediticio:

$$
\text{Protection Leg} = \int_{0}^{T} Z(\tau)(1 - R)dPD(\tau)\
$$

Donde:

+ $R$: Es el recovery rate, que representa la recuperación esperada del nominal por parte de la entidad emisora en caso de un default. Entonces $(1-R)$ Sería la pérdida esperada en caso de default.

+ $Z(\tau)$: Es el factor de descuento libre de riesgo en tiempo $\tau$.

+ $dPD(\tau)$: Es la probabilidad de superviviencia de la entidad en tiempo $\tau$.

(pag. 9)


### *¿Qué representa el spread de un CDS de manera matemática?, es decir, como se interpreta el ratio utilizado para calcularlo y que significa.*

Recordando la manera de calcular el spread de un CDS:

$$
S_0 = \frac{\text{Protection Leg}}{\text{RPV01}}
$$

Matemáticamente, el spread no es mas que una proporción, que nos dice que parte del protection leg representa el RPV01. Además es mencionado (pag. 11) que el spread es lo que logra el equilibrio para que el valor presente del premium y protection leg sean iguales. Para mejor despejamos y separamos de la siguiente manera:

$$
S_0 \cdot \text{RPV01} = \text{Protection Leg} 
$$

Tomando en cuenta este despeje, dado que el RPV01 es el valor presente de solo un punto base del premium leg, entonces, podemos intuir que el spread $S_0$ sería el costo en el premium leg que tendría que asumirse dado el riesgo de default, pues el protection leg depende del recovery rate, que captura directamente que tan riesgoso es el emisor. (pag. 11)

### *¿Qué otros elementos de un CDS es importante conocer y como se calculan?*

1. **Premium Leg**

Aunque ha ya ha sido mencionada a lo largo de esta investigación, formalmente, la definición de la premium leg son los flujos de pagos regulares del protection buyer al protección seller por el seguro ante un evento crediticio. Es calculado de la siguiente manera:

$$
\text{Premium Leg} = S_0 \cdot \text{RPV01}
$$

2. **Mark to Market**

En términos sencillos, es el P&L de la posición del CDS comparando el spread pactado vs el spread en el tiempo actual en el mercado. La forma de calcularlo es:

$$
\text{MTM} = (S_t - S_0) \text{RPV01}
$$

Donde:

+ $S_t$: es el spread actual en el mercado.

+ $S_0$: es el spread actual en pactado.

3. **Hazard Rates**

Los Hazard Rates son las probabilidades de que ocurra un evento crediticio en un intervalo de tiempo infitesimal $[t,t+dt)$, condicicionada a que la entidad ha sobrevivido hasta el tiempo $t$. (pag. 5). Los Hazard Rates son utilizados en los CDS para calcular la probabilidad de supervivencia de una entidad $Q(t_j)$, la cual es necesaria para el cálculo de el RPV01:

$$
Q(t_j) = e^{-\int_{t_v}^{T} \lambda(s)ds}
$$

Dónde:

+ $t_v$: es el tiempo en el que se realiza la valuación.

+ $\lambda$: es el hazard rate.

Sin embargo, puede haber diferenetes plazos en el CDS para los valores del spread (1Y, 3Y, 5Y, 7Y, 10Y), por lo que, se tiene que construir una curva de hazard rates. Para hacer esto se realiza el proceso iterativo *Bootstrapping*, el cual, consiste en obtener el hazard rate $\lambda_{0,1}$ y utilizarlo para calcular $\lambda_{1,3}$ y así sucesivamente. Asumiendo pagos de prima trimestrales, para el caso de un año se tendría que resolver:

$$
\begin{aligned}
\frac{S(t_V, t_V + 1Y)}{1 - R} \sum_{n=3,6,9,12} &\Delta(t_{n-3}, t_n, B) Z(t_V, t_n) e^{-\lambda_0 \tau_n}
&= \sum_{m=1}^{12} Z(t_V, t_m) \left(e^{-\lambda_0 \tau_{m-1}} - e^{-\lambda_0 \tau_m}\right)
\end{aligned}
$$

# **Referencias**

+ Hull, J., and A. White. “Valuing Credit Default Swaps I: No Counterparty Default Risk.” Journal of Derivatives. Vol. 8.

+ O'Kane, D. and S. Turnbull. “Valuation of Credit Default Swaps.” Lehman Brothers, Fixed Income Quantitative Credit Research, April 2003.

+ Beumee, J., D. Brigo, D. Schiemert, and G. Stoyle. “Charting a Course Through the CDS Big Bang.” Fitch Solutions, Quantitative
Research, Global Special Report. April 7, 2009.